# HuBMAP Vasculature — InternImage-T Cascade Inference (Kaggle)

mmdet v3 + InternImage-T(DCNv3_pytorch) + Cascade Mask R-CNN 單模推理 + 6 視角 TTA + morph 後處理。

## 使用流程
1. 把 `hubmap-internimage-cascade` Kaggle dataset 加進 notebook 的 input（包含 code/ configs/ weights/ wheels/）
2. 把 HuBMAP 比賽資料 `hubmap-hacking-the-human-vasculature` 加進 input
3. **Save & Run All（網路 ON）** 跑通驗證
4. 正式提交時 Settings → **Internet OFF** → Save & Run All（離線 wheel 安裝，等價於 ON）

## 警告
- 提交時 **必須關網**（HuBMAP 競賽規定）。所有依賴都已預裝在 dataset 的 `wheels/`，全程 `--no-index` 安裝。
- 若 Kaggle 升 torch / Python，需要重跑 `package_for_kaggle.sh --torch-version ... --cu-version ... --python-version ...` 重抓 wheel。


In [1]:
import sys, torch
print(f'Python: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}')
print(f'torch : {torch.__version__}')
print(f'cuda  : {torch.version.cuda}')


Python: 3.10.13
torch : 2.1.2
cuda  : 12.1


In [2]:
# ===== Cell 2: 偵測 Kaggle dataset 真實路徑 + symlink 成扁平 inputs/<slug>/ =====
# Kaggle 新版掛載格式：
#   /kaggle/input/datasets/<owner>/<slug>/   ← user dataset
#   /kaggle/input/competitions/<slug>/        ← competition data
#   /kaggle/input/<slug>/                     ← 舊版扁平掛載（forward-compat）
# 我們統一 symlink 到 /kaggle/working/inputs/<slug>/，後面 cell 用扁平 path。
import os
import glob

SYMLINK_DIR = '/kaggle/working/inputs'
os.makedirs(SYMLINK_DIR, exist_ok=True)

SLUG_TO_REAL = {}
for owner_dir in sorted(glob.glob('/kaggle/input/datasets/*')):
    if os.path.isdir(owner_dir):
        for ds in sorted(glob.glob(os.path.join(owner_dir, '*'))):
            if os.path.isdir(ds):
                SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/competitions/*')):
    if os.path.isdir(ds):
        SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/*')):
    name = os.path.basename(ds)
    if name in ('datasets', 'competitions'):
        continue
    if os.path.isdir(ds) and name not in SLUG_TO_REAL:
        SLUG_TO_REAL[name] = ds

for slug, real in SLUG_TO_REAL.items():
    link = os.path.join(SYMLINK_DIR, slug)
    if not os.path.exists(link):
        os.symlink(real, link)

print('偵測到的 datasets:')
for slug in sorted(SLUG_TO_REAL):
    print(f'  {slug:50s} -> {SLUG_TO_REAL[slug]}')

# 確認關鍵兩個 dataset 都在
PKG_SLUG = 'hubmap-internimage-cascade'      # ← 自己的權重 + wheels
COMP_SLUG = 'hubmap-hacking-the-human-vasculature'  # ← 比賽資料

assert PKG_SLUG in SLUG_TO_REAL, f'缺少 {PKG_SLUG} dataset，請把它 add to notebook input'
assert COMP_SLUG in SLUG_TO_REAL, f'缺少 {COMP_SLUG} 比賽資料，請 add the competition data'

PKG_DIR = f'/kaggle/working/inputs/{PKG_SLUG}'
COMP_DIR = f'/kaggle/working/inputs/{COMP_SLUG}'
print(f'\nPKG_DIR  = {PKG_DIR}')
print(f'COMP_DIR = {COMP_DIR}')

偵測到的 datasets:
  hubmap-hacking-the-human-vasculature               -> /kaggle/input/competitions/hubmap-hacking-the-human-vasculature
  hubmap-internimage-cascade                         -> /kaggle/input/datasets/paohuah/hubmap-internimage-cascade
  lal-deberta-base-mlm                               -> /kaggle/input/lal-deberta-base-mlm

PKG_DIR  = /kaggle/working/inputs/hubmap-internimage-cascade
COMP_DIR = /kaggle/working/inputs/hubmap-hacking-the-human-vasculature


In [3]:
# ===== Cell 3: 離線安裝（白名單過濾）+ mmcv 從本地原始碼 source build =====
# 為什麼科學 stack 不裝：Kaggle 預裝 numpy/scipy/...，硬蓋會炸 ABI。
# 為什麼 mmcv 要 source build：openmmlab 的 mmcv 2.1.0 cp310 wheel 是舊 GCC ABI 編的，
# Kaggle 的 torch 是新 ABI（_GLIBCXX_USE_CXX11_ABI=1），對不上會噴
#   undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationESs
# 解法：用 Kaggle 環境內的 gcc + torch headers 編一個對齊 ABI 的 _ext.so。
#
# ⚠️ 關鍵地雷：裝 mmcv 時「絕對不能讓 pip 去碰 numpy」。
#   --no-build-isolation 只管「build 依賴」，管不到 runtime 依賴！mmcv 2.1.0 的
#   install_requires 含 numpy；少了 --no-deps（網路 ON 時）pip 會去 PyPI 把 numpy 換版
#   → Kaggle 預裝的 scipy 是配原本那顆 numpy 編的，numpy 一換 scipy 的 compiled ufunc
#   ABI 立刻對不上，import albumentations 時噴：
#       ValueError: All ufuncs must have type `numpy.ufunc`.
#   這不是 mmcv 的 ABI 問題（mmcv build 其實成功了），是 numpy 被換掉害到 scipy。
#   => build 指令同時帶 --no-deps + --no-index，並在 build 前後 assert numpy 沒被動到。
#
# 離線 submit 友善：
#   - mmcv 原始碼已預先打包進 PKG_DIR/mmcv_src/（tarball 或 Kaggle 自動解壓的資料夾皆可）
#   - --no-build-isolation：不要再去抓 build 依賴（setuptools/wheel/packaging，
#     Kaggle base image 早就有）→ build 過程不需網路
import sys, os, subprocess, glob, importlib
from importlib.metadata import version as _pkg_version

# (0) Python cp 標籤
print(f'Kaggle Python: {sys.version_info.major}.{sys.version_info.minor}')

# (1) 環境健康檢查（source build 需要 nvcc + gcc）
import torch
print(f'torch: {torch.__version__}, cuda: {torch.version.cuda}, '
      f'cxx11_abi: {torch._C._GLIBCXX_USE_CXX11_ABI}')
ret = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
if ret.returncode != 0:
    raise RuntimeError('❌ nvcc 不存在，無法 source build mmcv。請確認 Kaggle GPU 環境。')
print('nvcc:', ret.stdout.strip().splitlines()[-1])

# 記下基準 numpy 版本（讀 dist metadata，不依賴 import 狀態），最後比對是否被換掉
_numpy_ver_before = _pkg_version('numpy')
print(f'numpy（基準，全程不可變動）: {_numpy_ver_before}')

# (2) 安裝非 mmcv 的白名單 wheel
WHEEL_DIR = f'{PKG_DIR}/wheels'
wheels = sorted(glob.glob(f'{WHEEL_DIR}/*.whl'))
INSTALL_PKGS = {
    'mmengine', 'mmdet', 'mmpretrain',           # mmcv 跳過 → 改 source build
    'albumentations', 'pycocotools',
    'addict', 'terminaltables', 'mat4py', 'einops',
    'ordered_set', 'ordered-set',
    'model_index', 'model-index', 'modelindex',
    'shapely', 'yapf', 'qudida',
}
def pkg_from_wheel(w):
    return os.path.basename(w).split('-')[0].lower()
wheels_to_install = [w for w in wheels if pkg_from_wheel(w) in INSTALL_PKGS]
print(f'\n[a] 裝白名單 wheel: {len(wheels_to_install)} 個（科學 stack + mmcv 跳過）')
cmd = ['pip', 'install', '-q', '--no-index', '--no-deps', '--find-links', WHEEL_DIR] + wheels_to_install
ret = subprocess.run(cmd, capture_output=True, text=True)
if ret.returncode != 0:
    print('STDERR:', ret.stderr[-2000:])
    raise RuntimeError(f'pip install 失敗 (exit {ret.returncode})')
print('✓ 白名單 wheel 安裝完成')

# (3) 找 mmcv 原始碼：相容「tarball」與「Kaggle 自動解壓成資料夾」兩種情況
#     上傳 .tar.gz 到 Kaggle dataset 時，Kaggle 常會自動解壓成 mmcv_src/mmcv-X.Y.Z/，
#     此時 glob *.tar.gz 會抓不到 → 退一步找含 setup.py 的原始碼資料夾。pip 兩種都能 build。
print('\n[b] Source build mmcv 從本地原始碼（~5 分鐘，無網路依賴）')
MMCV_SRC_DIR = f'{PKG_DIR}/mmcv_src'
cands = sorted(glob.glob(f'{MMCV_SRC_DIR}/mmcv-*.tar.gz'))                      # 情況 1：tarball 還在
if not cands:                                                                  # 情況 2：解壓成 mmcv-*/
    cands = sorted(d for d in glob.glob(f'{MMCV_SRC_DIR}/mmcv-*')
                   if os.path.isdir(d) and os.path.exists(os.path.join(d, 'setup.py')))
if not cands:                                                                  # 情況 3：多包一層 → 遞迴找 setup.py
    cands = sorted(os.path.dirname(p) for p in glob.glob(f'{MMCV_SRC_DIR}/**/setup.py', recursive=True))
assert cands, (
    f'❌ 在 {MMCV_SRC_DIR}/ 找不到 mmcv 原始碼（tarball 或解壓資料夾都沒有）。\n'
    f'   現有內容: {os.listdir(MMCV_SRC_DIR) if os.path.isdir(MMCV_SRC_DIR) else "(目錄不存在)"}\n'
    f'   請確認 dataset 裡有 mmcv_src/mmcv-*.tar.gz，或 Kaggle 解壓後的 mmcv-*/setup.py')
mmcv_src = cands[-1]
_kind = 'tarball' if mmcv_src.endswith('.tar.gz') else 'source dir'
print(f'使用 mmcv 原始碼（{_kind}）: {mmcv_src}')

# 關鍵：dataset 在 /kaggle/input（唯讀）。pip 對 source dir 做 in-place build 要在
# 原始碼目錄裡建 build/，唯讀 → 'could not create build: Read-only file system'。
# 解法：先把原始碼複製/解壓到可寫的 /kaggle/working/ 再 build。
import shutil, tarfile
BUILD_ROOT = '/kaggle/working/mmcv_build'
shutil.rmtree(BUILD_ROOT, ignore_errors=True)
os.makedirs(BUILD_ROOT, exist_ok=True)
if mmcv_src.endswith('.tar.gz'):
    with tarfile.open(mmcv_src) as tf:
        tf.extractall(BUILD_ROOT)
    _setups = sorted(os.path.dirname(p) for p in glob.glob(f'{BUILD_ROOT}/**/setup.py', recursive=True))
    assert _setups, f'tarball 解壓後找不到 setup.py: {os.listdir(BUILD_ROOT)}'
    build_src = _setups[0]
else:
    build_src = os.path.join(BUILD_ROOT, os.path.basename(mmcv_src.rstrip('/')))
    shutil.copytree(mmcv_src, build_src)
print(f'已複製到可寫目錄: {build_src}')

# 砍掉任何殘留的 mmcv（避免新舊版混雜）
subprocess.run(['pip', 'uninstall', '-y', 'mmcv', 'mmcv-full'], capture_output=True)

env = os.environ.copy()
env['MMCV_WITH_OPS'] = '1'
env['FORCE_CUDA'] = '1'
env['TORCH_CUDA_ARCH_LIST'] = '7.5'     # Kaggle T4 sm_75

# --no-build-isolation：不要再去抓 build 依賴（Kaggle 已有 setuptools/wheel）
# --no-deps      ：不要解析/安裝 mmcv 的執行期依賴（numpy/torch 已預裝，碰了會炸 ABI）
# --no-index     ：雙保險，禁止連 PyPI（就算哪天 --no-deps 被拿掉也不可能換 numpy）
# 加起來 = 完全離線、且保證不動科學 stack。pip 對 tarball 或資料夾都能 source build。
build_cmd = ['pip', 'install', '--no-build-isolation', '--no-deps', '--no-index', '-v', build_src]
print('執行:', ' '.join(build_cmd))
print('（編譯約 5 分鐘，只印最後 2000 字元）')
ret = subprocess.run(build_cmd, env=env, capture_output=True, text=True)
print(ret.stdout[-2000:] if ret.stdout else '')
if ret.returncode != 0:
    print('STDERR:', ret.stderr[-3000:])
    raise RuntimeError(f'mmcv source build 失敗 (exit {ret.returncode})')
print('✓ mmcv source build 完成')

# (3.5) 防呆：確認沒有任何 pip step 把 numpy 換掉（換掉就會炸 scipy ABI）
_numpy_ver_after = _pkg_version('numpy')
if _numpy_ver_after != _numpy_ver_before:
    raise RuntimeError(
        f'❌ numpy 被換版了：{_numpy_ver_before} → {_numpy_ver_after}\n'
        f'   這會讓 Kaggle 預裝的 scipy ABI 對不上（import albumentations 會噴\n'
        f'   "All ufuncs must have type numpy.ufunc"）。\n'
        f'   原因通常是某個 pip install 少了 --no-deps/--no-index 去 PyPI 換了 numpy。\n'
        f'   請 Factory reset kernel 後重跑（本 cell 已帶 --no-deps --no-index，不會再發生）。')
print(f'✓ numpy 全程未變動: {_numpy_ver_after}')

# (4) Patch mmdet 對 mmcv 的 runtime assert（mmcv 2.1.0 不會踩到，no-op）
spec = importlib.util.find_spec('mmdet')
if spec and spec.origin:
    with open(spec.origin) as f:
        src = f.read()
    src_patched = src.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
    if src_patched != src:
        with open(spec.origin, 'w') as f:
            f.write(src_patched)
        print(f'✓ patched mmdet: mmcv_maximum_version 2.2.0 → 2.3.0')

# (5) 驗證 import 都過得了
for mod in ['numpy', 'scipy', 'torch', 'mmengine', 'mmcv', 'mmcv.ops', 'mmdet', 'albumentations', 'pycocotools']:
    try:
        m = importlib.import_module(mod)
        print(f'  {mod:20s} = {getattr(m, "__version__", "?")}')
    except Exception as e:
        print(f'  {mod:20s} ❌ {type(e).__name__}: {str(e)[:120]}')

Kaggle Python: 3.10
torch: 2.1.2, cuda: 12.1, cxx11_abi: True
nvcc: Build cuda_12.1.r12.1/compiler.32688072_0
numpy（基準，全程不可變動）: 1.26.4

[a] 裝白名單 wheel: 15 個（科學 stack + mmcv 跳過）
✓ 白名單 wheel 安裝完成

[b] Source build mmcv 從本地原始碼（~5 分鐘，無網路依賴）
使用 mmcv 原始碼（source dir）: /kaggle/working/inputs/hubmap-internimage-cascade/mmcv_src/mmcv-2.1.0/mmcv-2.1.0
執行: pip install --no-build-isolation --no-deps --no-index -v /kaggle/working/inputs/hubmap-internimage-cascade/mmcv_src/mmcv-2.1.0/mmcv-2.1.0
（編譯約 5 分鐘，只印最後 2000 字元）
Using pip 23.3.2 from /opt/conda/lib/python3.10/site-packages/pip (python 3.10)
Processing ./inputs/hubmap-internimage-cascade/mmcv_src/mmcv-2.1.0/mmcv-2.1.0
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py clean for mmcv
Failed to build mmcv

STDERR: tps://setuptools.pypa.io/en/latest/pkg_resources.html
    from pkg_resources import DistributionNotFound, get_distribution, parse_version
  error: could not create 'bu

RuntimeError: mmcv source build 失敗 (exit 1)

In [ ]:
# ===== Cell 4: torch.load monkey-patch（與本地 train.py 同步）=====
# 為什麼需要：torch 2.6+ 把 torch.load(weights_only=True) 改成預設，
# 但 mmengine 0.10.x 的 load_from_local 沒傳 weights_only=False，會擋下
# ckpt 裡的 HistoryBuffer 等 mmengine 自家物件。我們載自己訓的 ckpt 是 trusted。
import torch

if not getattr(torch.load, '_hubmap_patched', False):
    _orig_torch_load = torch.load
    def _torch_load_patched(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(*args, **kwargs)
    _torch_load_patched._hubmap_patched = True
    torch.load = _torch_load_patched
    print('✓ torch.load patched (weights_only=False 預設)')
else:
    print('已 patched 過，跳過')


In [ ]:
# ===== Cell 5: 把 code/（含 models/）複製到 /kaggle/working/ 並加入 sys.path =====
# 為什麼複製不直接 sys.path inputs/：
#   - /kaggle/working/ 可寫，避免將來 mmdet/cython 想寫 __pycache__ 失敗
#   - subprocess 在 /kaggle/working/ 起來時找得到 models/
import sys
import shutil

WORK_CODE = '/kaggle/working/code'
if os.path.exists(WORK_CODE):
    shutil.rmtree(WORK_CODE)
shutil.copytree(f'{PKG_DIR}/code', WORK_CODE)

if WORK_CODE not in sys.path:
    sys.path.insert(0, WORK_CODE)

print(f'code/ 已複製到 {WORK_CODE}')
print('內容:')
for root, dirs, files in os.walk(WORK_CODE):
    rel = os.path.relpath(root, WORK_CODE)
    for f in sorted(files):
        if not f.endswith(('.pyc',)):
            print(f'  {os.path.join(rel, f) if rel != "." else f}')


In [ ]:
# ===== Cell 6: 子程序跑 kaggle_infer.py（隔離 Python state，避免污染 notebook kernel）=====
import subprocess

# 自動找 best ckpt
ckpts = sorted(__import__('glob').glob(f'{PKG_DIR}/weights/best_coco_segm_mAP_epoch_*.pth'))
assert ckpts, f'{PKG_DIR}/weights/ 沒有 best ckpt'
CKPT = ckpts[-1]
print(f'使用 ckpt: {os.path.basename(CKPT)}')

CONFIG = f'{PKG_DIR}/configs/stage2_dataset1_finetune.py'
IMG_DIR = f'{COMP_DIR}/test'
OUT_CSV = '/kaggle/working/submission.csv'
OUT_PKL = '/kaggle/working/internimage_preds.pkl'   # 給跨版本 4-stream WBF 用

cmd = [
    'python', f'{WORK_CODE}/kaggle_infer.py',
    '--config', CONFIG,
    '--checkpoint', CKPT,
    '--img-dir', IMG_DIR,
    '--out', OUT_CSV,
    '--pkl-out', OUT_PKL,    # 同時 dump ensemble pkl（keyed by img_id: rle+score+box）
    '--no-compile',          # Cascade RoI head 在 T4 上 torch.compile 易 graph break
    '--score-thr', '0.001',  # AP 指標必須保留低信心 TP（原本 0.3 是害你掉分的主因）
]
print('執行:', ' '.join(cmd))
print('-' * 60)

# 即時把子程序 stdout 串到 notebook（不要 capture_output 才能看 tqdm/print）
ret = subprocess.run(cmd, cwd=WORK_CODE)
if ret.returncode != 0:
    raise RuntimeError(f'kaggle_infer.py exit {ret.returncode}')

print('-' * 60)
print(f'✓ 推理完成 → {OUT_CSV}')
print(f'✓ ensemble pkl → {OUT_PKL}（把這個 notebook 的 Output 存成 dataset，餵給作者融合 notebook）')


In [ ]:
# ===== Cell 7: 檢查 submission.csv 格式 =====
import pandas as pd

OUT_CSV = '/kaggle/working/submission.csv'
df = pd.read_csv(OUT_CSV)
print(f'submission.csv: {len(df)} 列')
print()
print('欄位:', list(df.columns))
print()
print('前 5 列（pred_string 截短）:')
preview = df.copy()
preview['prediction_string'] = preview['prediction_string'].astype(str).str[:80] + '...'
print(preview.head().to_string(index=False))

# 驗證格式：欄位齊全、id 不重複、prediction_string 不為空（至少有 label=0 的開頭）
assert set(df.columns) == {'id', 'height', 'width', 'prediction_string'}, '欄位不對'
assert df['id'].is_unique, 'id 有重複'
n_empty = df['prediction_string'].fillna('').eq('').sum()
print(f'\n空 prediction_string 列數: {n_empty}（這些 tile 沒預測到 blood_vessel）')
